In [1]:
!pip install -q "numpy==1.26.4"

In [2]:
!pip install "transformer_lens==2.17.0"

In [3]:
import os
import json
import gc
import re
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.font_manager as fm
from transformer_lens import HookedTransformer
from transformers import AutoTokenizer
from pathlib import Path
from scipy import stats
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from collections import Counter
import warnings
import logging

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")
warnings.filterwarnings("ignore", message=".*Glyph.*missing.*")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

2026-05-27 17:20:58.498968: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779902458.710139     125 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779902458.768415     125 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779902459.265470     125 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779902459.265512     125 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779902459.265515     125 computation_placer.cc:177] computation placer alr

In [4]:
# ── configuration ───────────────────────────────────────────────
CLASSIFIED_FILE = '/kaggle/input/datasets/svvikhlyantseva/classified-corrections-expanded-context/classified_corrections_expanded_context.json'
SOURCE_FILE     = '/kaggle/input/datasets/svvikhlyantseva/merged-results/merged_results.json'
OUTPUT_DIR      = '/kaggle/working/logit_lens_output'
MODEL_NAME      = 'Qwen/Qwen3-8B'

MAX_TOKENS   = 256   # prompt truncation — keeps the end (closest to the marker)
N_TOP_TOKENS = 5     # top-k tokens to report per layer
RANDOM_SEED  = 42
# ────────────────────────────────────────────────────────────────

np.random.seed(RANDOM_SEED)
Path(OUTPUT_DIR).mkdir(exist_ok=True)

In [5]:
# ── 2. MODEL LOADING ────────────────────────────────────────────

def load_model():
    print(f"\nLoading model {MODEL_NAME}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    n_gpu  = torch.cuda.device_count()
    print(f"Device: {device}, GPUs: {n_gpu}")

    if n_gpu >= 2:
        model = HookedTransformer.from_pretrained(
            MODEL_NAME, device=None, n_devices=2,
            dtype=torch.float16, default_prepend_bos=False,
            fold_ln=False, center_writing_weights=False,
            move_to_device=True, trust_remote_code=True,
        )
    else:
        model = HookedTransformer.from_pretrained(
            MODEL_NAME, device=device,
            dtype=torch.float16, default_prepend_bos=False,
            fold_ln=False, center_writing_weights=False,
            trust_remote_code=True,
        )

    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

    print(f"Model: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")
    for i in range(n_gpu):
        print(f"GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

    return model, tokenizer


model, tokenizer = load_model()


Loading model Qwen/Qwen3-8B...
Device: cuda, GPUs: 2


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen3-8B into HookedTransformer
Model: 36 layers, d_model=4096
GPU 0: 8.29 GB
GPU 1: 8.29 GB


In [7]:
def generate_with_forced_close(model, tokenizer, prompt_text,
                                steering_vector=None, coeff=0,
                                steering_layer=None,
                                max_think_tokens=800,
                                max_answer_tokens=100,
                                temperature=0.6,
                                top_p=0.95):
    """
    Phase 1: steered thinking up to max_think_tokens.
    Phase 2: force </think> if not closed, then short answer phase.
    Steering stops after </think>.
    Temperature and top_p for sampling.
    """
    messages = [{"role": "user", "content": prompt_text}]
    prefix   = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    ) + "<think>\n"
    prompt_tokens = tokenizer.encode(prefix)[-256:]

    if (steering_vector is not None
            and coeff != 0
            and steering_layer is not None):
        layer_device = next(
            model.blocks[steering_layer].parameters()
        ).device
        sv_cpu    = torch.tensor(steering_vector, dtype=torch.float32)
        hook_name = f'blocks.{steering_layer}.hook_resid_post'
        use_hook  = True
    else:
        use_hook = False

    current       = torch.tensor(
        [prompt_tokens], dtype=torch.long, device='cuda:0'
    )
    thinking_ids  = []
    natural_close = False

    # ── Phase 1: thinking with sampling ──────────────────────────
    with torch.no_grad():
        for _ in range(max_think_tokens):
            if use_hook:
                sv_dev = sv_cpu.to(layer_device).to(torch.float16)

                def steering_hook(value, hook,
                                   _sv=sv_dev, _c=coeff):
                    _sv_local = _sv.to(value.device)
                    value = value.clone()
                    value[:, -1, :] += _c * _sv_local
                    return value

                logits = model.run_with_hooks(
                    current, fwd_hooks=[(hook_name, steering_hook)]
                )
            else:
                logits = model(current)

            # Apply temperature and top-p sampling
            logits_cpu = logits[0, -1, :].float().cpu()
            
            if temperature == 0:
                next_tok = int(torch.argmax(logits_cpu).item())
            else:
                # apply temperature
                logits_cpu = logits_cpu / temperature
                # apply top-p (nucleus sampling)
                sorted_logits, sorted_indices = torch.sort(logits_cpu, descending=True)
                cumulative_probs = torch.cumsum(
                    torch.softmax(sorted_logits, dim=-1), dim=-1
                )
                # remove tokens with cumulative probability above top_p
                sorted_indices_to_remove = cumulative_probs > top_p
                # shift right to keep first token above threshold
                sorted_indices_to_remove[..., 1:] = \
                    sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                logits_cpu[indices_to_remove] = float('-inf')
                # sample
                probs = torch.softmax(logits_cpu, dim=-1)
                next_tok = int(torch.multinomial(probs, num_samples=1).item())
            
            thinking_ids.append(next_tok)
            current = torch.cat([
                current,
                torch.tensor([[next_tok]], dtype=torch.long, device='cuda:0')
            ], dim=1)

            decoded = tokenizer.decode(thinking_ids)
            if '</think>' in decoded:
                natural_close = True
                break
            if next_tok == tokenizer.eos_token_id:
                break

    thinking_text = tokenizer.decode(thinking_ids)

    # ── Phase 2: force close + short answer (no steering) ─────────
    if not natural_close:
        close_ids = tokenizer.encode('</think>\n', add_special_tokens=False)
        current   = torch.cat([
            current,
            torch.tensor([close_ids], dtype=torch.long, device='cuda:0')
        ], dim=1)

    answer_ids = []
    with torch.no_grad():
        for _ in range(max_answer_tokens):
            # No steering in answer phase, but still use sampling
            logits = model(current)
            logits_cpu = logits[0, -1, :].float().cpu()
            
            if temperature == 0:
                next_tok = int(torch.argmax(logits_cpu).item())
            else:
                logits_cpu = logits_cpu / temperature
                sorted_logits, sorted_indices = torch.sort(logits_cpu, descending=True)
                cumulative_probs = torch.cumsum(
                    torch.softmax(sorted_logits, dim=-1), dim=-1
                )
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = \
                    sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                logits_cpu[indices_to_remove] = float('-inf')
                probs = torch.softmax(logits_cpu, dim=-1)
                next_tok = int(torch.multinomial(probs, num_samples=1).item())
            
            answer_ids.append(next_tok)
            current = torch.cat([
                current,
                torch.tensor([[next_tok]], dtype=torch.long, device='cuda:0')
            ], dim=1)
            if next_tok == tokenizer.eos_token_id:
                break

    answer_text = tokenizer.decode(answer_ids)
    return thinking_text, answer_text, natural_close

In [8]:
# ════════════════════════════════════════════════════════════════
# Helper functions
# ════════════════════════════════════════════════════════════════

def count_correction_markers(text, markers=None):
    """Count self-correction markers in generated text."""
    if markers is None:
        markers = ['wait', 'actually', 'let me reconsider', 'correction',
                   'sorry', 'i meant', 'my mistake', 'on second thought',
                   'hang on', 'hold on', "that's not right", 'i made an error']
    text_lower = text.lower()
    counts = {m: text_lower.count(m) for m in markers if text_lower.count(m) > 0}
    return counts, sum(counts.values())


def extract_aime_answer(answer_text, question_text=''):
    """Extract numeric answer from AIME response."""
    q_nums = set(re.findall(r'\b\d+\b', question_text))
    boxed = re.findall(r'\\boxed\{(\d+)\}', answer_text)
    if boxed:
        return int(boxed[-1])
    for pat in [r'(?:answer is|=)\s*\**(\d+)\**',
                r'm\s*\+\s*n\s*=\s*(\d+)',
                r'therefore[,\s]+(\d+)']:
        found = re.findall(pat, answer_text.lower())
        if found:
            c = int(found[-1])
            if 0 <= c <= 999:
                return c
    nums = re.findall(r'\b(\d{1,3})\b', answer_text)
    for num in reversed(nums):
        c = int(num)
        if str(c) not in q_nums and 0 <= c <= 999:
            return c
    return None


def is_clean_prompt(q_text):
    """Exclude questions with complex LaTeX diagrams."""
    bad_patterns = [
        r'\\begin\{tikzpicture\}',
        r'\\coordinate',
        r'\\draw\[',
        r'\\begin\{align',
    ]
    for pat in bad_patterns:
        if re.search(pat, q_text):
            return False
    return True


def norm(v):
    return v / (np.linalg.norm(v) + 1e-8)

In [9]:
# ════════════════════════════════════════════════════════════════
# Load vectors and compute similarities
# ════════════════════════════════════════════════════════════════

saved = np.load('/kaggle/input/datasets/svvikhlyantseva/correction-vectors-all-methods-extended/correction_vectors_all_methods_extended.npz')

STEERING_LAYER = 21

raw_vec       = saved['mean_raw']
dom_vec       = saved['dom_combined'][STEERING_LAYER]
error_cf_vec  = saved['mean_error_cf']
all_wait_vec  = saved['mean_all_wait']

# Normalise all to unit norm
sv_genuine   = norm(raw_vec.astype(np.float32))
sv_dom       = norm(dom_vec.astype(np.float32))
sv_error_cf  = norm(error_cf_vec.astype(np.float32))
sv_all_wait  = norm(all_wait_vec.astype(np.float32))

np.random.seed(RANDOM_SEED)
sv_rand = norm(np.random.randn(model.cfg.d_model).astype(np.float32))

print("Cosine similarities (sanity check):")
vectors_to_compare = [
    ('genuine', sv_genuine),
    ('all_wait', sv_all_wait),
    ('dom_combined', sv_dom),
    ('error_cf', sv_error_cf),
    ('random', sv_rand)
]

for i, (n1, v1) in enumerate(vectors_to_compare):
    for n2, v2 in vectors_to_compare[i+1:]:
        s = float(cosine_similarity(v1.reshape(1,-1), v2.reshape(1,-1))[0,0])
        print(f"  {n1} vs {n2}: {s:.4f}")

Cosine similarities (sanity check):
  genuine vs all_wait: 0.1935
  genuine vs dom_combined: 0.5361
  genuine vs error_cf: 0.6528
  genuine vs random: 0.0041
  all_wait vs dom_combined: 0.1702
  all_wait vs error_cf: 0.0751
  all_wait vs random: 0.0103
  dom_combined vs error_cf: 0.3606
  dom_combined vs random: -0.0029
  error_cf vs random: 0.0045


In [11]:
# ════════════════════════════════════════════════════════════════
# Load and filter AIME questions
# ════════════════════════════════════════════════════════════════

# Load AIME dataset
from datasets import load_dataset

aime_ds = load_dataset("MathArena/aime_2025", split="train")
aime_lookup = {
    ' '.join(item['problem'].strip().split()): int(item['answer'])
    for item in aime_ds
}

# Load your source file
with open(SOURCE_FILE, 'r', encoding='utf-8') as f:
    source = json.load(f)

GROUND_TRUTH = {}
math_questions = []
for q in source['questions']:
    if q.get('category') != 'math_trick':
        continue
    q_norm = ' '.join(q['question'].strip().split())
    if q_norm in aime_lookup:
        GROUND_TRUTH[q['id']] = aime_lookup[q_norm]
        math_questions.append(q)

print(f"\nTotal matched AIME questions: {len(math_questions)}")

# Filter clean prompts and take first 5 for testing
AIME_SUBSET = [
    q for q in math_questions
    if is_clean_prompt(q['question'])
][:5]  # 5 questions for quick testing

print(f"Clean AIME questions used: {len(AIME_SUBSET)}")
for q in AIME_SUBSET:
    print(f"  Q{q['id']}: {q['question'][:60]}...")

data/train-00000-of-00001.parquet:   0%|          | 0.00/14.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30 [00:00<?, ? examples/s]


Total matched AIME questions: 30
Clean AIME questions used: 5
  Q3: Let $A$ be the set of positive integer divisors of $2025$. L...
  Q4: Let $A_1 A_2 A_3 \ldots A_{11}$ be an $11$-sided non-convex ...
  Q12: Alex divides a disk into four quadrants with two perpendicul...
  Q13: Find the sum of all positive integers $n$ such that $n+2$ di...
  Q22: There are $8!= 40320$ eight-digit positive integers that use...


In [12]:
# ════════════════════════════════════════════════════════════════
# Main experiment: 3 runs per condition
# ════════════════════════════════════════════════════════════════

COEFF_POS = 40
MAX_THINK_TOKENS = 800
MAX_ANSWER_TOKENS = 100
TEMPERATURE = 0.6
TOP_P = 0.95
NUM_RUNS = 3  # 3 repeats per condition

# Define vectors to test
AIME_VECTORS = {
    'genuine_hc': sv_genuine,
    'all_wait': sv_all_wait,
    'dom_combined': sv_dom,
    'error_cf': sv_error_cf,
    'random': sv_rand,
}

# Results structure: {vector_name: {question_id: {run_idx: {'markers': x, 'tokens': y, 'pred': z}}}}
results_aime = {vn: {} for vn in list(AIME_VECTORS.keys()) + ['baseline']}
aime_records = []

print("\n" + "="*80)
print(f"EXPERIMENT: AIME 2025")
print(f"  coeff={COEFF_POS} | layer={STEERING_LAYER}")
print(f"  questions={len(AIME_SUBSET)} | runs per condition={NUM_RUNS}")
print(f"  temperature={TEMPERATURE} | top_p={TOP_P}")
print(f"  max_think_tokens={MAX_THINK_TOKENS} | max_answer_tokens={MAX_ANSWER_TOKENS}")
print(f"  vectors: {list(AIME_VECTORS.keys())}")
print("="*80)

for p_idx, q in enumerate(AIME_SUBSET):
    q_id = q['id']
    q_text = q['question']
    gt = GROUND_TRUTH[q_id]
    
    print(f"\n{'='*80}")
    print(f"[Question {p_idx+1}/{len(AIME_SUBSET)}] Q{q_id} | GT={gt}")
    print(f"  {q_text[:100]}...")
    print(f"{'='*80}")
    
    question_record = {
        'question_id': q_id,
        'gt': gt,
        'question': q_text,
        'runs': {}
    }
    
    # For each condition (baseline + all vectors)
    for cond_name in ['baseline'] + list(AIME_VECTORS.keys()):
        print(f"\n  >>> Condition: {cond_name}")
        
        cond_runs = []
        
        for run_idx in range(NUM_RUNS):
            print(f"    Run {run_idx + 1}/{NUM_RUNS}: ", end='', flush=True)
            
            torch.cuda.empty_cache()
            gc.collect()
            
            # Determine steering parameters
            if cond_name == 'baseline':
                sv = None
                coeff = 0
                layer = None
            else:
                sv = AIME_VECTORS[cond_name]
                coeff = COEFF_POS
                layer = STEERING_LAYER
            
            try:
                thinking, answer, natural_close = generate_with_forced_close(
                    model, tokenizer, q_text,
                    steering_vector=sv,
                    coeff=coeff,
                    steering_layer=layer,
                    max_think_tokens=MAX_THINK_TOKENS,
                    max_answer_tokens=MAX_ANSWER_TOKENS,
                    temperature=TEMPERATURE,
                    top_p=TOP_P
                )
                
                # Count markers
                _, n_markers = count_correction_markers(thinking)
                n_tokens = len(tokenizer.encode(thinking))
                
                # Extract answer
                pred = extract_aime_answer(answer, q_text)
                correct = int(pred == gt) if pred is not None else 0
                
                print(f"markers={n_markers}, tokens={n_tokens}, pred={pred}, correct={correct}")
                
                cond_runs.append({
                    'run': run_idx,
                    'markers': n_markers,
                    'tokens': n_tokens,
                    'predicted': pred,
                    'correct': correct,
                    'closed': natural_close,
                    'thinking_preview': thinking[:300],
                    'answer_preview': answer[:200]
                })
                
            except Exception as e:
                print(f"ERROR: {e}")
                cond_runs.append({
                    'run': run_idx,
                    'markers': 0,
                    'tokens': 0,
                    'predicted': None,
                    'correct': 0,
                    'closed': False,
                    'error': str(e)
                })
        
        # Store results for this condition
        results_aime[cond_name][q_id] = cond_runs
        question_record['runs'][cond_name] = cond_runs
        
        # Print summary for this condition
        markers_vals = [r['markers'] for r in cond_runs]
        correct_vals = [r['correct'] for r in cond_runs]
        print(f"    Summary: markers={np.mean(markers_vals):.1f}±{np.std(markers_vals):.1f}, "
              f"accuracy={np.mean(correct_vals):.2f}")
    
    aime_records.append(question_record)
    
    # Save intermediate results
    with open(f'{OUTPUT_DIR}/aime_5questions_3runs.json', 'w') as f:
        json.dump(aime_records, f, ensure_ascii=False, indent=2)


EXPERIMENT: AIME 2025
  coeff=40 | layer=21
  questions=5 | runs per condition=3
  temperature=0.6 | top_p=0.95
  max_think_tokens=800 | max_answer_tokens=100
  vectors: ['genuine_hc', 'all_wait', 'dom_combined', 'error_cf', 'random']

[Question 1/5] Q3 | GT=237
  Let $A$ be the set of positive integer divisors of $2025$. Let $B$ be a randomly selected subset of ...

  >>> Condition: baseline
    Run 1/3: markers=2, tokens=800, pred=None, correct=0
    Run 2/3: markers=5, tokens=800, pred=None, correct=0
    Run 3/3: markers=2, tokens=800, pred=2, correct=0
    Summary: markers=3.0±1.4, accuracy=0.00

  >>> Condition: genuine_hc
    Run 1/3: markers=7, tokens=800, pred=1, correct=0
    Run 2/3: markers=6, tokens=800, pred=15, correct=0
    Run 3/3: markers=9, tokens=800, pred=None, correct=0
    Summary: markers=7.3±1.2, accuracy=0.00

  >>> Condition: all_wait
    Run 1/3: markers=4, tokens=800, pred=15, correct=0
    Run 2/3: markers=4, tokens=800, pred=None, correct=0
    Run 3/3: 

KeyboardInterrupt: 

In [ ]:
# ════════════════════════════════════════════════════════════════
# Statistics and Visualization
# ════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("FINAL RESULTS - 3 runs per condition, 5 questions")
print("="*80)

# Aggregate statistics
all_vectors = ['baseline'] + list(AIME_VECTORS.keys())

print("\nMean markers (±std) across all runs and questions:")
for vn in all_vectors:
    all_markers = []
    for q_id, runs in results_aime[vn].items():
        all_markers.extend([r['markers'] for r in runs])
    mean_m = np.mean(all_markers)
    std_m = np.std(all_markers)
    print(f"  {vn:>15}: {mean_m:.2f} ± {std_m:.2f}")

# Accuracy
print("\nAccuracy (correct answer extraction):")
for vn in all_vectors:
    all_correct = []
    for q_id, runs in results_aime[vn].items():
        all_correct.extend([r['correct'] for r in runs])
    acc = np.mean(all_correct)
    print(f"  {vn:>15}: {acc:.3f} ({sum(all_correct)}/{len(all_correct)})")

# Statistical tests - compare all vectors with random
rand_markers = []
for q_id, runs in results_aime['random'].items():
    rand_markers.extend([r['markers'] for r in runs])

print("\nMann-Whitney U (H1: vector > random, markers):")
for vn in ['genuine_hc', 'all_wait', 'dom_combined', 'error_cf']:
    vec_markers = []
    for q_id, runs in results_aime[vn].items():
        vec_markers.extend([r['markers'] for r in runs])
    stat, p = stats.mannwhitneyu(vec_markers, rand_markers, alternative='greater')
    sig = 'p<0.05 ✓' if p < 0.05 else 'not significant'
    print(f"    {vn:>15}: p={p:.4f}  {sig}")

# Comparison between vectors
print("\nPairwise comparisons (H1: genuine_hc > others):")
for vn in ['all_wait', 'dom_combined', 'error_cf']:
    genuine_markers = []
    other_markers = []
    for q_id, runs in results_aime['genuine_hc'].items():
        genuine_markers.extend([r['markers'] for r in runs])
    for q_id, runs in results_aime[vn].items():
        other_markers.extend([r['markers'] for r in runs])
    stat, p = stats.mannwhitneyu(genuine_markers, other_markers, alternative='greater')
    sig = 'p<0.05 ✓' if p < 0.05 else 'not significant'
    print(f"    genuine_hc vs {vn:>15}: p={p:.4f}  {sig}")


# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar plot with error bars (aggregated across all runs)
means_m = []
stds_m = []
for vn in all_vectors:
    all_markers = []
    for q_id, runs in results_aime[vn].items():
        all_markers.extend([r['markers'] for r in runs])
    means_m.append(np.mean(all_markers))
    stds_m.append(np.std(all_markers))

colors = ['#2c3e50', '#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#95a5a6']
bars = axes[0].bar(all_vectors, means_m, color=colors, alpha=0.85,
                   yerr=stds_m, capsize=5, edgecolor='white')
axes[0].set_ylabel('Mean correction markers', fontsize=11)
axes[0].set_title(f'All vector types: marker count\n'
                  f'AIME 2025, n={len(AIME_SUBSET)} questions × {NUM_RUNS} runs',
                  fontsize=10)
axes[0].tick_params(axis='x', rotation=15)

# Per-question boxplot
all_data = []
for vn in all_vectors:
    vn_markers = []
    for q_id, runs in results_aime[vn].items():
        vn_markers.extend([r['markers'] for r in runs])
    all_data.append(vn_markers)

bp = axes[1].boxplot(all_data, labels=all_vectors, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
axes[1].set_ylabel('Correction markers', fontsize=11)
axes[1].set_title('Distribution across all runs', fontsize=10)
axes[1].tick_params(axis='x', rotation=15)

# Accuracy bar plot
accuracies = []
for vn in all_vectors:
    all_correct = []
    for q_id, runs in results_aime[vn].items():
        all_correct.extend([r['correct'] for r in runs])
    accuracies.append(np.mean(all_correct))

bars2 = axes[2].bar(all_vectors, accuracies, color=colors, alpha=0.85, edgecolor='white')
axes[2].set_ylabel('Accuracy', fontsize=11)
axes[2].set_ylim(0, 1)
axes[2].set_title('Answer extraction accuracy', fontsize=10)
axes[2].tick_params(axis='x', rotation=15)
for bar, val in zip(bars2, accuracies):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f'{val:.2f}', ha='center', fontsize=9)

plt.suptitle(f'Positive steering — comparison of all vector types (3 runs each)\n'
             f'Qwen3-8B | layer {STEERING_LAYER} | coeff={COEFF_POS} | T={TEMPERATURE}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/aime_all_vectors_3runs.png', dpi=130, bbox_inches='tight')
plt.show()
plt.close()

print(f"\nResults saved to: {OUTPUT_DIR}/aime_5questions_3runs.json")